In [ ]:
import pandas as pd
import unicodedata

# Load TSV/CSV
df = pd.read_csv("data/sinhala-sentiment-analysis/test.tsv", sep="\t")

# Strip whitespace from column names
df.columns = df.columns.str.strip()

# Select only needed columns
df = df[['title', 'body', 'comment_phrase', 'comment_sentiment']]

# Clean text
for col in ['title', 'body', 'comment_phrase']:
    df[col] = df[col].astype(str).str.replace('"', '').str.strip()
    df[col] = df[col].apply(lambda x: unicodedata.normalize('NFC', x) if isinstance(x, str) and x != 'nan' else x)

# Drop exact duplicates
df = df.drop_duplicates()

# Save cleaned CSV
df.to_csv("cleaned_dataset_test.csv", index=False)

In [ ]:
out = pd.read_csv("outputs/train.csv")

In [ ]:
print(out.count())

In [ ]:
print(out['comment_sentiment'].value_counts())

In [ ]:
print("Total rows:", len(out))

In [ ]:
print(out.isnull().sum())

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv(
    "cleaned_dataset_test.csv",
    engine="python",
    on_bad_lines="skip"
)

# Keep only needed columns
df = df[['title', 'body', 'comment_phrase', 'comment_sentiment']]

# Keep rows where sentiment exists
df = df[df['comment_sentiment'].notna()]

# Remove empty strings
df = df[df['comment_sentiment'].str.strip() != ""]

# Reset index
df = df.reset_index(drop=True)

# Save cleaned dataset
df.to_csv("filtered_dataset_test.csv", index=False)

# Print stats
print("Rows kept:", len(df))
print(df['comment_sentiment'].value_counts())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load dataset with robust parameters
df = pd.read_csv(
    "filtered_dataset_train.csv",
    engine="python",
    on_bad_lines="skip"
)

# Strip whitespace from column names
df.columns = df.columns.str.strip()

print("Columns in dataset:")
print(df.columns.tolist())
print(f"\nDataset shape: {df.shape}")

# Remove rows where body is missing
df = df[df["body"].notna()]

# Compute word counts
df["body_word_count"] = df["body"].astype(str).apply(lambda x: len(x.split()))

# Statistics
avg_words = df["body_word_count"].mean()
min_words = df["body_word_count"].min()
max_words = df["body_word_count"].max()

print(f"\nAverage word count: {avg_words:.2f}")
print(f"Minimum word count: {min_words}")
print(f"Maximum word count: {max_words}")

# Histogram bins (auto-scale using range)
bins = 30

# Plot histogram
plt.figure(figsize=(10,6))
plt.hist(df["body_word_count"], bins=bins)
plt.xlabel("Word Count in Body")
plt.ylabel("Number of Articles")
plt.title("Distribution of Article Body Word Counts")
plt.show()

In [ ]:
import pandas as pd

# Load datasets
df1 = pd.read_csv("filtered_dataset_train.csv")
df2 = pd.read_csv("filtered_dataset_test.csv")

combined_df = pd.concat([df1, df2], ignore_index=True)

# remove duplicates
combined_df = combined_df.drop_duplicates()

# remove empty rows
combined_df = combined_df.dropna()

# shuffle
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Combined dataset shape:", combined_df.shape)
print(combined_df["comment_sentiment"].value_counts())

In [ ]:
import pandas as pd

# Load dataset

df = pd.read_csv("combined_shuffled_dataset.csv")

# Clean column names

df.columns = df.columns.str.strip()

# Remove rows where body is missing

df = df[df["body"].notna()]

# Compute word count for body

df["body_word_count"] = df["body"].astype(str).apply(lambda x: len(x.split()))

# Define bins

bins = [0, 50, 100, 200, 400, 800, 1600]
labels = ["0-50", "50-100", "100-200", "200-400", "400-800", "800-1600"]

# Assign bins

df["word_bin"] = pd.cut(df["body_word_count"], bins=bins, labels=labels)

# Print dataset info

print("\nDataset shape:", df.shape)

# Print distribution per sentiment category

for sentiment in sorted(df["comment_sentiment"].unique()):
    print(f"\nSentiment: {sentiment}")

    subset = df[df["comment_sentiment"] == sentiment]

    counts = subset["word_bin"].value_counts().sort_index()
    print("\nWord count distribution:")
    print(counts)

    avg = subset["body_word_count"].mean()
    min_wc = subset["body_word_count"].min()
    max_wc = subset["body_word_count"].max()

    print("\nStatistics:")
    print(f"Average word count: {avg:.2f}")
    print(f"Minimum word count: {min_wc}")
    print(f"Maximum word count: {max_wc}")
